In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType
import ast

In [0]:
display(spark.sql("SELECT * FROM bronze_games LIMIT 5"))

# check total rows
display(spark.sql("""
SELECT 
  COUNT(*) as total_rows,
  COUNT(DISTINCT name) as unique_games
FROM bronze_games
"""))

# check duplicates
display(spark.sql("""
SELECT name, COUNT(*) as count 
FROM bronze_games 
GROUP BY name 
HAVING COUNT(*) > 1
"""))

Databricks data profile. Run in Databricks to view.

In [0]:
@udf(ArrayType(StringType()))
def parse_pylist(s): # receives one single cell value from one row. Spark calls this function once per row
    if s is None:
        return None
    try:
        result = ast.literal_eval(s)
        return result if isinstance(result, list) else None
    except (ValueError, SyntaxError):
        return None

# parse the three array columns that broke FROM_JSON, expose as a temp view
df = spark.table("bronze_games")
df = (df
    .withColumn("genres", parse_pylist("genres"))
    .withColumn("developer", parse_pylist("developer"))
    .withColumn("publisher", parse_pylist("publisher")))

df.createOrReplaceTempView("bronze_games_parsed")

In [0]:
%sql
CREATE OR REPLACE TABLE silver_games AS
WITH parsed AS (
  SELECT
    LOWER(TRIM(REGEXP_REPLACE(name, '[®™©]', ''))) AS game_key,
    name,
    short_description,
    long_description,
    genres,
    COALESCE(
      TRY_TO_DATE(release_date, 'd MMM, yyyy'),
      TRY_TO_DATE(release_date, 'MMM yyyy')
    ) AS release_date,
    developer,
    publisher,
    CASE overall_player_rating
      WHEN 'Overwhelmingly Positive' THEN 7
      WHEN 'Very Positive' THEN 6
      WHEN 'Positive' THEN 5
      WHEN 'Mostly Positive' THEN 4
      WHEN 'Mixed' THEN 3
      WHEN 'Mostly Negative' THEN 2
      WHEN 'Very Negative' THEN 1
    END AS rating_score,
    overall_player_rating AS rating_label,
    CASE WHEN number_of_reviews_from_purchased_people LIKE '%of%'
      THEN TRY_CAST(ROUND(
        TRY_CAST(REGEXP_EXTRACT(number_of_reviews_from_purchased_people, '(\\d+)') AS FLOAT) / 100 *
        TRY_CAST(REPLACE(REGEXP_EXTRACT(number_of_reviews_from_purchased_people, 'of ([\\d,]+)'), ',', '') AS INT)
      ) AS INT)
      ELSE TRY_CAST(REGEXP_EXTRACT(REPLACE(number_of_reviews_from_purchased_people, ',', ''), '(\\d+)') AS INT)
    END AS reviews_purchased,
    TRY_CAST(REPLACE(number_of_english_reviews, ',', '') AS INT) AS reviews_english,
    link
  FROM bronze_games_parsed
)
SELECT
  *,
  CASE WHEN reviews_english > reviews_purchased THEN 1 ELSE 0 END AS review_count_inconsistent
FROM parsed;

In [0]:
# check null values for important columns
display(spark.sql("""SELECT
  COUNT(*) - COUNT(name) as null_name,
  COUNT(*) - COUNT(release_date) as null_release_date,
  COUNT(*) - COUNT(reviews_purchased) as null_reviews_purchased,
  COUNT(*) - COUNT(rating_score) as null_rating
FROM silver_games"""))

display(spark.sql("""
SELECT name, reviews_purchased, rating_score
FROM silver_games
WHERE reviews_purchased IS NULL OR rating_score IS NULL
"""))

In [0]:
%sql
CREATE OR REPLACE TABLE silver_games AS
SELECT *
FROM silver_games
WHERE reviews_purchased IS NOT NULL 
  AND reviews_purchased > 0;

SELECT
  COUNT(*)          AS total,
  COUNT(genres)     AS genres_ok,
  COUNT(developer)  AS dev_ok,
  COUNT(publisher)  AS pub_ok,
  COUNT(rating_score) AS rating_ok,
  COUNT(reviews_purchased) AS purch_ok,
  COUNT(reviews_english)   AS eng_ok
FROM silver_games

In [0]:
%sql
SELECT *
FROM silver_games